In [1]:
!pip install hdbscan

In [2]:
import numpy as np
import pandas as pd
import hdbscan
import plotly.express as px
from sklearn.preprocessing import StandardScaler

# 1. SYNTHETIC DATA GENERATION (Gandhipuram Scenario)
base_lat, base_lon = 11.0180, 76.9660

np.random.seed(42)
n_main = 200
main_lat = np.random.normal(base_lat, 0.002, n_main)
main_lon = np.random.normal(base_lon, 0.002, n_main)
main_time = np.random.uniform(0, 30, n_main) 

n_detour = 15
detour_lat = np.random.normal(base_lat + 0.004, 0.0002, n_detour)
detour_lon = np.random.normal(base_lon + 0.004, 0.0002, n_detour)
detour_time = np.random.uniform(14, 16, n_detour) 

n_slowdrip = 10
slowdrip_lat = np.random.normal(base_lat + 0.004, 0.0002, n_slowdrip)
slowdrip_lon = np.random.normal(base_lon + 0.004, 0.0002, n_slowdrip)
slowdrip_time = np.linspace(0, 30, n_slowdrip) 

n_rogues = 5
rogue_lat = np.random.normal(base_lat, 0.005, n_rogues)
rogue_lon = np.random.normal(base_lon, 0.005, n_rogues)
rogue_time = np.random.uniform(0, 30, n_rogues)

lats = np.concatenate([main_lat, detour_lat, slowdrip_lat, rogue_lat])
lons = np.concatenate([main_lon, detour_lon, slowdrip_lon, rogue_lon])
times = np.concatenate([main_time, detour_time, slowdrip_time, rogue_time])
types = ['Main']*n_main + ['Detour']*n_detour + ['Slow-Drip']*n_slowdrip + ['Rogue']*n_rogues

df = pd.DataFrame({'Latitude': lats, 'Longitude': lons, 'Time_Min': times, 'True_Type': types})

# 2. SPATIAL-TEMPORAL CLUSTERING (HDBSCAN)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[['Latitude', 'Longitude', 'Time_Min']])

# FIX 1: Multiply the Time dimension weight by 5 so HDBSCAN strictly cuts clusters separated by time
X_scaled[:, 2] *= 5.0 

clusterer = hdbscan.HDBSCAN(min_cluster_size=10, metric='euclidean')
df['Cluster'] = clusterer.fit_predict(X_scaled)

# 3. BACKEND LOGIC GATES (The System Log)
print("=== REDTAXI BACKEND SYSTEM LOG ===")
print("Analyzing unmapped spatial deviations...\n")

try:
    detour_cluster_id = df[(df['True_Type'] == 'Detour') & (df['Cluster'] != -1)]['Cluster'].mode()[0]
    detour_data = df[df['Cluster'] == detour_cluster_id]
    car_count = len(detour_data)
    time_window = detour_data['Time_Min'].max() - detour_data['Time_Min'].min()
except IndexError:
    car_count = 0
    time_window = 999
    detour_cluster_id = -999

cond1_pass = car_count >= 10
print(f"Condition 1 [Spatial Density]: {car_count} vehicles detected. " + ("[PASS]" if cond1_pass else "[FAIL]"))

cond2_pass = time_window <= 3.0
print(f"Condition 2 [Temporal Window]: {time_window:.2f} minutes. " + ("[PASS]" if cond2_pass else "[FAIL]"))

mock_main_road_speed = 5 
cond3_pass = mock_main_road_speed < 15
print(f"Condition 3 [Velocity Cross-Check]: Main road speed {mock_main_road_speed} km/h. " + ("[PASS]" if cond3_pass else "[FAIL]"))

# FIX 2: Strict Boolean AND Gate forcing all three conditions to be TRUE
if cond1_pass and cond2_pass and cond3_pass:
    print("\n[VERDICT]: REROUTE VALIDATED. Muting penalty beeps and updating Navigation SDK.")
else:
    print("\n[VERDICT]: REROUTE REJECTED. Conditions not met or suspected fraud.")

# 4. VISUALIZATION
def rename_labels(row):
    if row['Cluster'] == -1: return 'Noise (Beep Active - Scams & Rogues)'
    elif row['Cluster'] == detour_cluster_id and cond1_pass and cond2_pass and cond3_pass: return 'Validated Detour (Beep Muted)'
    else: return 'Main Route Traffic / Rejected Detour'

df['Map_Label'] = df.apply(rename_labels, axis=1)

fig = px.scatter_mapbox(df, 
                        lat="Latitude", 
                        lon="Longitude", 
                        color="Map_Label",
                        color_discrete_map={
                            'Noise (Beep Active - Scams & Rogues)': 'black',
                            'Main Route Traffic / Rejected Detour': 'red',
                            'Validated Detour (Beep Muted)': 'blue'
                        },
                        hover_data=['Time_Min'],
                        zoom=14, 
                        mapbox_style="open-street-map")

fig.update_layout(title="RedTaxi Spatial-Temporal Anomaly Detection (HDBSCAN)",
                  margin={"r":0,"t":40,"l":0,"b":0})
fig.show()

=== REDTAXI BACKEND SYSTEM LOG ===
Analyzing unmapped spatial deviations...

Condition 1 [Spatial Density]: 18 vehicles detected. [PASS]
Condition 2 [Temporal Window]: 2.60 minutes. [PASS]
Condition 3 [Velocity Cross-Check]: Main road speed 5 km/h. [PASS]

[VERDICT]: REROUTE VALIDATED. Muting penalty beeps and updating Navigation SDK.
